# The Ultimate Async Data Stream (Capstone)

Difficulty: Advanced | ~50 min | Capstone: requires Labs 1-5

A real-time data ingestion engine: a slow paginated API is streamed into local
storage piece by piece. Every advanced feature in this catalog - async
generators, the event loop, decorators, lambda pipelines, and context-managed
resources - shows up in one production-style pipeline.

In [1]:
!pip install tabulate==0.10.0


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\Dell\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


## Section 0 - The slow paginated source

`fetch_page(page)` simulates a slow paginated API: 50 ms of fake network
time per page, 10 pages x 10 events = 100 total.

In [2]:
import asyncio
import functools
import json
import time

PAGE_SIZE = 10
TOTAL_PAGES = 10

# simulated paginated API: 50 ms latency, deterministic events per page
async def fetch_page(page):
    await asyncio.sleep(0.05)  # 50 ms of fake network latency per page
    events = [
        {
            "event_id": (page - 1) * PAGE_SIZE + i,
            "kind": "purchase" if i % 3 == 0 else "click",
            "amount": i * 7 + page,
        }
        for i in range(PAGE_SIZE)
    ]
    return {"page": page, "events": events, "has_more": page < TOTAL_PAGES}

## Section 1 - @timer, extended for coroutines

Lab 4's `@timer` is reused with one addition: it must also time coroutine
functions. The decorator checks `asyncio.iscoroutinefunction(func)` and
returns an `async def wrapper` that `await`s the call.

In [3]:
# detect coroutine function at decoration time, not call time
def timer(func):
    if asyncio.iscoroutinefunction(func):
        # async branch: wrapper must await to actually run the coroutine
        @functools.wraps(func)
        async def wrapper(*args, **kwargs):
            start = time.perf_counter()
            result = await func(*args, **kwargs)
            print(f"{func.__name__} ran in {time.perf_counter() - start:.2f}s")
            return result
        return wrapper
    else:
        # sync branch: plain call, same timing logic
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            start = time.perf_counter()
            result = func(*args, **kwargs)
            print(f"{func.__name__} ran in {time.perf_counter() - start:.2f}s")
            return result
        return wrapper

## Section 2 - The async generator

`async def` + `yield` gives a lazy stream that awaits a slow source. Each
`await fetch_page(page)` suspends the generator so the loop can work meanwhile;
`yield` hands one page to the consumer and pauses until the next is requested.

In [4]:
# async generator: await a slow page, yield it, repeat until done
async def stream_pages():
    page = 1
    while True:
        data = await fetch_page(page)  # suspend; the loop works meanwhile
        yield data                      # hand this page to the consumer now
        if not data["has_more"]:
            return
        page += 1

## Section 3 - Proving the laziness

Pulling two pages then `break`ing costs `2 x 50 ms` - the generator stops
mid-stream and never requests page 3. An eager list would have paid the full
10-page latency before the first yield.

In [5]:
# count how many pages we actually consume; break after 2 to prove laziness
start = time.perf_counter()
seen = 0
async for page in stream_pages():
    seen += 1
    if seen == 2:
        break
print(f"consumed {seen}/{TOTAL_PAGES} pages in {time.perf_counter() - start:.2f}s")

consumed 2/10 pages in 0.13s


## Section 4 - The lambda cleaning pipeline

Lab 1's `filter` + `map` composition, applied on the fly: the moment a page
arrives, `filter` drops events below a threshold and `map` attaches `price`.

In [6]:
# filter: keep events with amount >= 30; map: add price = amount * 1.08
def clean(events):
    return list(map(
        lambda ev: {**ev, "price": round(ev["amount"] * 1.08, 2)},
        filter(lambda ev: ev["amount"] >= 30, events)))

## Section 5 - The context-managed store

Lab 2's `__enter__`/`__exit__` as async protocol: `__aenter__`/`__aexit__`
awaited by `async with`. The exit method runs on success and on crash.

In [7]:
# async context manager: __aenter__ opens, __aexit__ always closes
class LocalStore:
    def __init__(self, path):
        self.path = path
    async def __aenter__(self):
        self.file = open(self.path, "w", encoding="utf-8")
        self.rows = 0
        return self
    async def __aexit__(self, exc_type, exc, tb):
        self.file.close()  # runs on success AND on exception
        if exc_type is None:
            print(f"committed {self.rows} rows to {self.path}")
        else:
            print(f"closed {self.path} after {exc_type.__name__}")
    def write(self, events):
        for ev in events:
            self.file.write(json.dumps(ev) + "\n")
            self.rows += 1
        self.file.flush()  # flush after each batch so data is on disk

## Section 6 - The capstone: wiring it all together

Every piece of the catalog in one function:

- `@timer` - benchmarks the entire run.
- `async with LocalStore(...)` - resource opened and guaranteed closed.
- `async for page in stream_pages()` - generator streams one page at a time.
- `clean(page["events"])` - filter + map pipeline scrubs each page on arrival.

In [8]:
# capstone: timer + async with + async for + clean, all in one function
@timer
async def ingest():
    fetched = 0  # total events off the wire
    stored = 0   # events that survived the filter
    async with LocalStore("events.jsonl") as db:
        async for page in stream_pages():        # lazy: one page at a time
            fetched += len(page["events"])       # every event off the wire
            events = clean(page["events"])       # filter + map on arrival
            db.write(events)
            stored += len(events)
            print(f"page {page['page']}: kept {len(events)}/{len(page['events'])}")
    return fetched, stored

fetched, stored = await ingest()
print("fetched:", fetched, "stored:", stored)

page 1: kept 5/10
page 2: kept 6/10
page 3: kept 6/10
page 4: kept 6/10
page 5: kept 6/10
page 6: kept 6/10
page 7: kept 6/10
page 8: kept 6/10
page 9: kept 7/10
page 10: kept 7/10
committed 61 rows to events.jsonl
ingest ran in 0.63s
fetched: 100 stored: 61


## Section 7 - Reading the ledger

The stored file is read back and cross-checked against the counters: every
kept event must be on disk, one JSON per line, nothing dropped.

In [9]:
from tabulate import tabulate

# read back JSONL and cross-check against pipeline counters
with open("events.jsonl", encoding="utf-8") as f:
    rows = [json.loads(line) for line in f]

# every kept event must be on disk, nothing dropped
assert len(rows) == stored == 61
assert fetched == TOTAL_PAGES * PAGE_SIZE

print("first stored:", rows[0])
print("last stored:", rows[-1])
# summary table: fetched vs kept vs stored
ledger = [
    ["fetched (events)", fetched, f"{TOTAL_PAGES} pages x {PAGE_SIZE}"],
    ["kept (amount >= 30)", stored, f"{stored}/{fetched} survived the filter"],
    ["stored (JSON lines)", len(rows), "events.jsonl, one JSON per line"],
]
print(tabulate(ledger, headers=["stage", "count", "what happened"], tablefmt="grid"))

first stored: {'event_id': 5, 'kind': 'click', 'amount': 36, 'price': 38.88}
last stored: {'event_id': 99, 'kind': 'purchase', 'amount': 73, 'price': 78.84}
+---------------------+---------+---------------------------------+
| stage               |   count | what happened                   |
+=====================+=========+=================================+
| fetched (events)    |     100 | 10 pages x 10                   |
+---------------------+---------+---------------------------------+
| kept (amount >= 30) |      61 | 61/100 survived the filter      |
+---------------------+---------+---------------------------------+
| stored (JSON lines) |      61 | events.jsonl, one JSON per line |
+---------------------+---------+---------------------------------+


## Section 8 - Optional Exercise: make it resilient

Real ingestion engines fail, and this exercise makes yours fail - on purpose.

1. Swap the source for a flaky twin: redefine `fetch_page` to raise
   `RuntimeError("gateway timed out")` on `page % 4 == 0`, but only the first
   time that page is requested (track hit pages in a set so the failure is
   intermittent, not guaranteed - a guaranteed failure is a broken API, and
   retrying it forever is pointless).
2. Make the generator resilient: wrap `await fetch_page(page)` in a retry loop
   that attempts the page up to 3 times with `await asyncio.sleep(0.02)` between
   tries, and re-raises only after the budget is spent (Lab 4's `@retry` idea,
   inlined because it must live inside the generator).
3. Re-run `ingest()`. Pages 4 and 8 must be recovered on the retry, the store
   must still print `committed 61 rows`, and `events.jsonl` must still hold all
   61 lines.
4. Now prove the crash guarantee: raise `ValueError` inside an `async with
   LocalStore(...)` block after writing a few rows, catch it, and confirm the
   exit message reports the crash and the file is still readable (the partial
   line is corruption you can see - the open file handle is not).

Make the pipeline resilient, then prove its crash guarantee.

1. Swap the source for a flaky twin: redefine `fetch_page` to raise `RuntimeError("gateway timed out")` on `page % 4 == 0`, but only the *first* time that page is requested — track already-failed pages in a set, so the failure is intermittent. (A guaranteed failure is a broken API; retrying it forever is pointless.)
2. Make the generator resilient: replace the single `await fetch_page(page)` with a retry loop that tries up to 3 times, sleeping `await asyncio.sleep(0.02)` between attempts, and re-raises only after the budget is spent. (Lab 4's `@retry` idea, inlined because the loop must live inside the generator.)
3. Re-run `ingest()`. Pages 4 and 8 must be recovered on the retry, the store must still print `committed 61 rows`, and `events.jsonl` must still hold all 61 lines.
4. Prove the crash guarantee: raise a `ValueError` inside an `async with LocalStore(...)` block after writing a few rows, catch it, and confirm the exit message reports the crash and the file is still readable.